<a href="https://colab.research.google.com/github/linhkid/pallas-forge/blob/main/notebooks/05_reproduce_figures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — Reproduce the figures with real TPU data

The seven PNGs in `images/` that ship with this repo are generated from a **synthetic cost model** (`scripts/generate_example_images.py`). They're useful placeholders while developing on CPU — but for the blog posts, the README, and anything public-facing, you want **real numbers from a real TPU**.

This notebook:
1. Installs pallas-forge on a Colab TPU runtime
2. Runs the three reference kernels with realistic sizes
3. Regenerates `heatmap_matmul.png`, `heatmap_tflops.png`, `roofline.png`, `speedup_vs_xla.png` — the figures that go into the blog/README
4. Zips them so you can download with one click

**Runtime requirements:** `Runtime → Change runtime type → TPU`. A free Colab TPU v2 works fine; v5e on Colab Pro is faster and closer to the numbers in the sprint's target hardware.

**Expected wall time:** ~5-10 minutes end-to-end.

## Setup

In [1]:
# Install JAX with TPU support + pallas-forge
#
# Two install paths:
#   A) from GitHub tag v0.1.2 (default — requires the repo to be pushed)
#   B) from an uploaded wheel in /content/ — fallback if GitHub isn't ready
#
# To use (B): build locally with `python -m build`, upload the wheel via the
# Files panel on the left, then set INSTALL_FROM = "wheel" below.

import os
import subprocess
import sys

INSTALL_FROM = "github"  # or "wheel"
GITHUB_URL = "pallas-forge[viz] @ git+https://github.com/linhkid/pallas-forge.git@v0.1.2"
WHEEL_PATH = "/content/pallas_forge-0.1.2-py3-none-any.whl"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Uninstall any broken prior attempt (no-op on first run)
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "pallas-forge"],
        capture_output=True,
    )

    # jax[tpu] — required for real TPU execution
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "jax[tpu]",
            "-f",
            "https://storage.googleapis.com/jax-releases/libtpu_releases.html",
        ]
    )

    # pallas-forge
    if INSTALL_FROM == "wheel":
        assert os.path.exists(WHEEL_PATH), (
            f"Upload the wheel to {WHEEL_PATH} first — use the Files panel on the left."
        )
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", f"{WHEEL_PATH}[viz]"]
        )
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", GITHUB_URL])

    # Verify the install actually took — pip can return 0 without the module being importable
    try:
        import pallas_forge  # noqa: F401
    except ImportError as e:
        raise RuntimeError(
            f"pallas_forge install did not take ({e}). Check that:\n"
            f"  - If INSTALL_FROM='github': the repo is pushed and the v0.1.2 tag exists\n"
            f"  - If INSTALL_FROM='wheel': {WHEEL_PATH} exists (Files panel on the left)\n"
            f"  - You may need to Runtime > Restart session after a failed install"
        )
    print(f"✓ pallas_forge {pallas_forge.__version__} installed from {INSTALL_FROM}")
else:
    print("Not running on Colab — assuming pallas-forge is already installed.")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


✓ pallas_forge 0.1.2 installed from github


In [2]:
# Verify TPU is attached
import jax

devices = jax.devices()
print(f"Devices ({len(devices)}): {[d.platform for d in devices]}")

tpu_platform = devices[0].platform
if tpu_platform != "tpu":
    raise RuntimeError(
        "This notebook requires a TPU runtime. "
        "Go to Runtime → Change runtime type → TPU."
    )

# Detect which TPU generation is attached (best-effort — Colab exposes this inconsistently)
TPU_GEN = "v5e"  # safe default; override below if detectable
try:
    from pallas_forge.profile import TPU_SPECS
    # Try to pull from the device kind string
    kind = str(getattr(devices[0], "device_kind", "")).lower()
    for k in ("v5p", "v5e", "v4"):
        if k in kind:
            TPU_GEN = k
            break
    print(f"Detected device kind: {kind or 'unknown'} — using TPU_SPECS['{TPU_GEN}']")
except Exception as e:
    print(f"(Couldn't auto-detect TPU kind, defaulting to {TPU_GEN}): {e}")

tpu = TPU_SPECS[TPU_GEN]
print(f"Peak: {tpu['peak_tflops_bf16']} TFLOPS bf16, {tpu['peak_bandwidth_gb_s']} GB/s HBM")

Devices (1): ['tpu']
Detected device kind: tpu v5 lite — using TPU_SPECS['v5e']
Peak: 197.0 TFLOPS bf16, 819.0 GB/s HBM


In [3]:
import os
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import jax
import jax.numpy as jnp

from pallas_forge import tiled_matmul, fused_rmsnorm_residual, fused_swiglu
from pallas_forge.tune import tune, TuneConfig
from pallas_forge.tune.runner import BenchmarkRunner
from pallas_forge.profile import roofline_chart

OUT = Path("/content/images_real") if IN_COLAB else Path("images")
OUT.mkdir(parents=True, exist_ok=True)
print(f"Outputs will be written to: {OUT}")

Outputs will be written to: /content/images_real


## Figure 1 & 2 — MatMul heatmaps (latency + throughput)

A block-size sweep on a 2048 × 2048 × 2048 bf16 matmul. `4 × 4 × 4 = 64` configurations, with 5 warmup + 15 timed iterations each.

In [4]:
M = K = N = 2048

def mm_input_fn(cfg):
    k1, k2 = jax.random.split(jax.random.PRNGKey(0))
    x = jax.random.normal(k1, (M, K), dtype=jnp.bfloat16)
    w = jax.random.normal(k2, (K, N), dtype=jnp.bfloat16)
    return (x, w)

def mm_kernel_fn(x, w, *, block_m, block_k, block_n):
    return tiled_matmul(x, w, block_m=block_m, block_k=block_k, block_n=block_n)

mm_config = TuneConfig.from_dict({
    "block_m": [64, 128, 256, 512],
    "block_k": [64, 128, 256, 512],
    "block_n": [64, 128, 256, 512],
})

mm_report = tune(
    kernel_fn=mm_kernel_fn,
    input_fn=mm_input_fn,
    config=mm_config,
    n_warmup=5,
    n_repeat=15,
    flops_fn=lambda _: 2 * M * K * N,
    bytes_fn=lambda _: (M * K + K * N + M * N) * 2,  # bf16 = 2 bytes
    verbose=True,
)

Auto-tuning 64 configurations...
[1/64] Benchmarking: {'block_m': 64, 'block_k': 64, 'block_n': 64}
  -> FAILED: The Pallas TPU lowering currently requires that the last two dimensions of your block shape are divisible by 8 and 128 respectively, or be equal to the respective dimensions of the overall array. Block spec for args[0] in pallas_call _matmul_kernel at /usr/local/lib/python3.12/dist-packages/pallas_forge/kernels/matmul.py:32 has block shape (Blocked(block_size=64), Blocked(block_size=64)), array shape (2048, 2048), and index_map { lambda ; a:i32[] b:i32[] c:i32[]. let  in (a, c) }, in memory space None.
See details at https://docs.jax.dev/en/latest/pallas/grid_blockspec.html#pallas-blockspec
[2/64] Benchmarking: {'block_m': 64, 'block_k': 64, 'block_n': 128}
  -> FAILED: The Pallas TPU lowering currently requires that the last two dimensions of your block shape are divisible by 8 and 128 respectively, or be equal to the respective dimensions of the overall array. Block spec f

In [5]:
# Heatmap 1: latency
fig = mm_report.heatmap(
    "block_m", "block_n",
    metric="median_ms",
    title=f"MatMul latency on TPU {TPU_GEN}  —  {M}×{K}×{N}  (bf16)",
    cmap="YlOrRd_r",
    figsize=(9, 7),
)
fig.savefig(OUT / "heatmap_matmul.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("  ✓ heatmap_matmul.png")

# Heatmap 2: throughput
fig = mm_report.heatmap(
    "block_m", "block_n",
    metric="tflops",
    title=f"MatMul throughput (TFLOPS)  —  {M}² bf16, TPU {TPU_GEN}",
    cmap="YlGn",
    figsize=(9, 7),
)
fig.savefig(OUT / "heatmap_tflops.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("  ✓ heatmap_tflops.png")

best = mm_report.best(1)[0]
print(f"\nBest matmul config: {best.config}")
print(f"  → {best.median_ms:.2f} ms, {best.tflops:.1f} TFLOPS")
print(f"Speedup range across sweep: {mm_report.speedup_range:.1f}×")

  ✓ heatmap_matmul.png
  ✓ heatmap_tflops.png

Best matmul config: {'block_m': 512, 'block_k': 512, 'block_n': 512}
  → 0.32 ms, 54.1 TFLOPS
Speedup range across sweep: 8.4×


## Figure 3 — Roofline with all 3 reference kernels

Benchmark the other two kernels at a single good config each, combine with the best matmul result, place on the roofline chart.

In [6]:
# --- RMSNorm benchmark (memory-bound) ---
BATCH, SEQ, DIM = 4, 2048, 4096

def rms_input_fn(_):
    k1, k2, k3 = jax.random.split(jax.random.PRNGKey(1), 3)
    x = jax.random.normal(k1, (BATCH, SEQ, DIM), dtype=jnp.bfloat16)
    res = jax.random.normal(k2, (BATCH, SEQ, DIM), dtype=jnp.bfloat16)
    w = jax.random.normal(k3, (DIM,), dtype=jnp.bfloat16)
    return (x, res, w)

def rms_kernel_fn(x, res, w):
    out, _ = fused_rmsnorm_residual(x, res, w)
    return out

tokens = BATCH * SEQ
rms_flops = tokens * DIM * 5  # rough: add + square + mean + div + mul
rms_bytes = (tokens * DIM * 2 + DIM + tokens * DIM * 2) * 2  # read x + res + weight, write out + new_res

rms_runner = BenchmarkRunner(
    rms_kernel_fn, rms_input_fn,
    n_warmup=5, n_repeat=15,
    flops_fn=lambda _: rms_flops,
    bytes_fn=lambda _: rms_bytes,
)
rms_result = rms_runner.run_single({})
rms_result.config = {"kernel": "fused_rmsnorm"}
print(f"RMSNorm: {rms_result.median_ms:.2f} ms, {rms_result.bandwidth_gb_s:.0f} GB/s")

RMSNorm: 0.89 ms, 303 GB/s


In [7]:
# --- SwiGLU benchmark (compute-bound) ---
BS, D, FFN = 2048, 4096, 11008  # LLaMA-7B-ish

def sg_input_fn(_):
    k1, k2, k3 = jax.random.split(jax.random.PRNGKey(2), 3)
    x = jax.random.normal(k1, (BS, D), dtype=jnp.bfloat16)
    wg = jax.random.normal(k2, (D, FFN), dtype=jnp.bfloat16)
    wu = jax.random.normal(k3, (D, FFN), dtype=jnp.bfloat16)
    return (x, wg, wu)

def sg_kernel_fn(x, wg, wu):
    return fused_swiglu(x, wg, wu, block_m=128, block_n=256)

sg_flops = 2 * BS * D * FFN * 2 + BS * FFN * 6  # two matmuls + activation
sg_bytes = (BS * D + D * FFN * 2 + BS * FFN) * 2

sg_runner = BenchmarkRunner(
    sg_kernel_fn, sg_input_fn,
    n_warmup=5, n_repeat=15,
    flops_fn=lambda _: sg_flops,
    bytes_fn=lambda _: sg_bytes,
)
sg_result = sg_runner.run_single({})
sg_result.config = {"kernel": "fused_swiglu"}
print(f"SwiGLU:  {sg_result.median_ms:.2f} ms, {sg_result.tflops:.1f} TFLOPS")

SwiGLU:  4.22 ms, 87.6 TFLOPS


In [8]:
# Roofline chart with all three kernels
best_mm = mm_report.best(1)[0]
best_mm.config = {"kernel": "tiled_matmul (best)"}

kernels = [best_mm, rms_result, sg_result]
labels = [r.config["kernel"] for r in kernels]

fig = roofline_chart(
    kernels,
    peak_tflops=tpu["peak_tflops_bf16"],
    peak_bandwidth_gb_s=tpu["peak_bandwidth_gb_s"],
    title=f"Roofline — pallas-forge reference kernels on TPU {TPU_GEN}",
    labels=labels,
    figsize=(10, 7),
)
fig.savefig(OUT / "roofline.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("  ✓ roofline.png")

  ✓ roofline.png


## Figure 4 — Speedup vs XLA baseline

For each kernel, benchmark the unfused XLA equivalent and compare.

In [9]:
# XLA baselines
def xla_matmul(x, w):
    return jnp.matmul(x, w)

def xla_rmsnorm(x, res, w, eps=1e-6):
    new_res = x + res
    var = jnp.mean(new_res.astype(jnp.float32) ** 2, axis=-1, keepdims=True)
    return ((new_res.astype(jnp.float32) / jnp.sqrt(var + eps)) * w.astype(jnp.float32)).astype(x.dtype)

def xla_swiglu(x, wg, wu):
    return jax.nn.silu(jnp.matmul(x, wg)) * jnp.matmul(x, wu)

xla_mm = BenchmarkRunner(xla_matmul, lambda _: mm_input_fn({}),
                         n_warmup=5, n_repeat=15).run_single({})
xla_rms = BenchmarkRunner(xla_rmsnorm, rms_input_fn,
                          n_warmup=5, n_repeat=15).run_single({})
xla_sg = BenchmarkRunner(xla_swiglu, sg_input_fn,
                         n_warmup=5, n_repeat=15).run_single({})

pallas_times = [best_mm.median_ms, rms_result.median_ms, sg_result.median_ms]
xla_times = [xla_mm.median_ms, xla_rms.median_ms, xla_sg.median_ms]
kernel_names = ["Tiled\nMatMul", "Fused\nRMSNorm+Res", "Fused\nSwiGLU"]
speedups = [x / p for x, p in zip(xla_times, pallas_times)]

print("\n  Kernel          XLA (ms)   Pallas (ms)   Speedup")
print("  " + "-" * 55)
for name, xt, pt, sp in zip(kernel_names, xla_times, pallas_times, speedups):
    print(f"  {name.replace(chr(10), ' '):<15} {xt:>8.2f}   {pt:>10.2f}   {sp:>6.2f}×")


  Kernel          XLA (ms)   Pallas (ms)   Speedup
  -------------------------------------------------------
  Tiled MatMul        0.24         0.32     0.77×
  Fused RMSNorm+Res     3.05         0.89     3.44×
  Fused SwiGLU        2.76         4.22     0.65×


In [10]:
# Speedup bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [3, 2]})
x = np.arange(len(kernel_names))
width = 0.35
ax1.bar(x - width / 2, xla_times, width, label="XLA baseline", color="#8b9dc3")
ax1.bar(x + width / 2, pallas_times, width, label="pallas-forge (best)", color="#ea4335")
ax1.set_ylabel("Median latency (ms)")
ax1.set_title(f"Per-kernel latency: Pallas-forge vs XLA baseline (TPU {TPU_GEN})")
ax1.set_xticks(x); ax1.set_xticklabels(kernel_names); ax1.legend()
ax1.grid(axis="y", alpha=0.3)

colors = ["#ea4335" if s >= 1.0 else "#8b9dc3" for s in speedups]
bars = ax2.bar(x, speedups, color=colors)
ax2.axhline(1.0, color="black", linestyle="--", alpha=0.5, label="XLA parity")
ax2.set_ylabel("Speedup vs XLA (×)")
ax2.set_title("Speedup factor")
ax2.set_xticks(x); ax2.set_xticklabels(kernel_names)
ax2.set_ylim(0, max(max(speedups) * 1.25, 1.25))
ax2.grid(axis="y", alpha=0.3)
for bar, sp in zip(bars, speedups):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
             f"{sp:.2f}×", ha="center", fontweight="bold")

plt.tight_layout()
fig.savefig(OUT / "speedup_vs_xla.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("  ✓ speedup_vs_xla.png")

  ✓ speedup_vs_xla.png


## Save raw results + download

In [11]:
# Also save the CSV/JSON so the numbers are auditable
mm_report.to_csv(OUT / "matmul_results.csv")
mm_report.to_json(OUT / "matmul_results.json")
print("  ✓ matmul_results.csv + .json")

print(f"\nAll outputs in {OUT}:")
for f in sorted(OUT.iterdir()):
    kb = f.stat().st_size / 1024
    print(f"  {f.name:<30} ({kb:.1f} KB)")

  ✓ matmul_results.csv + .json

All outputs in /content/images_real:
  heatmap_matmul.png             (53.5 KB)
  heatmap_tflops.png             (52.2 KB)
  matmul_results.csv             (5.3 KB)
  matmul_results.json            (11.9 KB)
  roofline.png                   (107.2 KB)
  speedup_vs_xla.png             (69.8 KB)


In [12]:
# Zip for easy download (Colab only)
if IN_COLAB:
    import shutil
    zip_path = shutil.make_archive(
        base_name="/content/pallas_forge_real_figures",
        format="zip",
        root_dir=str(OUT),
    )
    print(f"Bundle: {zip_path}")
    from google.colab import files
    files.download(zip_path)
else:
    print("Local run — files are already on disk.")

Bundle: /content/pallas_forge_real_figures.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What to do with the download

1. Unzip `pallas_forge_real_figures.zip` locally
2. Copy the PNGs into the repo's `images/` directory (replacing the synthetic ones)
3. Update the caption in `README.md` from "Illustrative numbers" to "Measured on TPU {TPU_GEN}"
4. Commit — now the figures in the repo, blog, and LinkedIn post reflect real hardware

### Re-running with a different TPU

Each TPU generation (v2, v3, v4, v5e, v5p, Trillium) will produce noticeably different numbers. For the blog series, the cleanest story is to **pick one generation and label the figures clearly.** If you only have free Colab access, v2 is fine — just rename the images to `heatmap_matmul_v2.png` etc. so readers know which hardware produced them.

### Caveats

- **Colab TPUs are shared.** Timing variance is higher than on a dedicated TPU VM. Increase `n_repeat` to 25-30 for publishable numbers.
- **First run is slow.** JIT compilation is not in the timed path (warmup covers it), but the first kernel can still take 20-30 s to launch.
- **No guarantee you get a TPU.** On free Colab, TPU availability fluctuates. If the runtime assignment fails, retry or try again later.